In [1]:
%load_ext autoreload
%autoreload 2

In [2]:

from bokeh.layouts import row, column
from bokeh.models import Button, WMTSTileSource, ColumnDataSource, Slider, CustomJS, HoverTool, CheckboxGroup, ColorPicker, TextInput, Div, AutocompleteInput
from bokeh.plotting import figure, show

import os


from manager import SourceManager, DynamicSourceManager

import sys
sys.path.insert(0, '..')
from src.load_data import DataSet
from src.tiles import ImageTiles
from src.utils.bins2spots import Bins2Spots
from src.sample import Bins, Spots

In [3]:
from bokeh.io import output_notebook
output_notebook()

Loading BokehJS ...

In [4]:
INITIAL_COLOR = '#6baed6'
INITIAL_ALPHA = 0.5

In [5]:
dir_base = '/home/lalonso/PhD/VisHD/data/'
dataset = DataSet(dir_base=dir_base, img_file="Visium_HD_Mouse_Small_Intestine_tissue_image.btf", bins_size=16)
img, adata, df_tissue_positions = dataset.load()
print("Data loaded succesfully")

/home/lalonso/PhD/VisHD/.venv/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/lalonso/PhD/VisHD/.venv/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Data loaded succesfully


In [20]:
image_tiles = ImageTiles(img, os.path.join(dir_base + "./tiles/"))
port = 4000
image_tiles.deploy_server(port=port)
print(f"Tiles server deployed on port {port}")

 * Serving Flask app 'TileSource'
 * Debug mode: off
Tiles server deployed on port 4000


Address already in use
Port 4000 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [21]:
max_dim = max(img.shape[:2])
background_color = 'white'
p = figure(x_range=(0, max_dim), y_range=(0, max_dim), background_fill_color=background_color)
p.grid.visible = False
p.xaxis.visible = False
p.yaxis.visible = False
# p.x_range.bounds = (0, max_dim)
# p.y_range.bounds = (0, max_dim)

In [22]:
custom_tile = WMTSTileSource(
    url= f"http://localhost:{port}" + "/{Z}/{X}/{Y}",  # Ensure port matches deployed tile server
    tile_size=256,
    max_zoom=image_tiles.max_zoom-1,
    min_zoom=0,
    wrap_around=False,
    x_origin_offset=0,
    y_origin_offset=0,
    initial_resolution=max_dim / image_tiles.tile_size
)
p.add_tile(custom_tile)

TileRenderer(id='p1629', ...)

In [23]:
img_overlay_source = ColumnDataSource({
    'x': [max_dim / 2],
    'y': [max_dim / 2],
    'width': [max_dim],
    'height': [max_dim],
    'alpha': [0.0]  # Initial transparent state
})
overlay = p.rect('x', 'y', 'width', 'height', source=img_overlay_source, color=background_color, alpha='alpha')

# Create a slider to control the alpha of the image overlay
img_alpha_slider = Slider(start=0.0, end=1.0, value=1.0, step=0.01, title="Image alpha")

# JavaScript to update the alpha of the overlay rectangle
update_img_alpha = CustomJS(args=dict(source=img_overlay_source), code="""
    const data = source.data;
    const alpha = cb_obj.value;
    data['alpha'][0] = 1 - alpha;
    source.change.emit();
""")
img_alpha_slider.js_on_change('value', update_img_alpha)

In [24]:
b2s = Bins2Spots(adata, df_tissue_positions, bins_size=dataset.bins_size)
INITIAL_GENE = adata.var_names.tolist()[2]

## Gene input
gene_pool = adata.var_names.tolist() + adata.obs.columns.tolist()
gene_input = AutocompleteInput(title="Gene/Metadata", value=INITIAL_GENE, completions=gene_pool, min_characters=2, case_sensitive=False, restrict=True)

In [25]:
from parameter import Scaler, StaticParameter

spots = Spots(
    b2s.spot_polygons, 
    b2s.convert(normalize=False), 
    tile_scale_xfactor=image_tiles.scale_xfactor,
    tile_scale_yfactor=image_tiles.scale_yfactor)
spots_manager = SourceManager(spots)
spots_manager.add_parameter(Scaler(0, 1, 
                                   INITIAL_GENE, 
                                   attr='alpha', 
                                   static_not_sequenced=True,
                                   not_sequenced_val=1.0
                                ))
spots_manager.add_parameter(Scaler(0.5, 2, 
                                   INITIAL_GENE, 
                                   attr='line_width', 
                                   static_not_sequenced=True,
                                   not_sequenced_val=0
                                ))
spots_manager.add_parameter(StaticParameter(1.0, attr='line_alpha', not_sequenced_val=0.0))
spots_manager.add_parameter(StaticParameter(INITIAL_COLOR, attr='color'))
spots_manager.fetch_data()

radius = b2s.converter.get_pixels(b2s.converter.VisiumV2.SPOT_DIAMETER) / 2

circle_glyph = p.circle(
    x='x', 
    y='y', 
    color='color', 
    radius=radius, 
    fill_alpha='alpha', 
    line_width='line_width',
    source=spots_manager.source,
    hover_color='pink'
)

## Hover 
hover = HoverTool(tooltips=[("Spot", "x: @x, y: @y, a: @alpha")], renderers=[circle_glyph])
p.add_tools(hover)

## Alpha slider
spots_alpha_slider = Slider(start=0.0, end=1.0, value=1.0, step=0.05, title="Spots alpha")

def update_spots_alpha(attr, old, new):
    print("Changing alpha")
    def weight_alpha():
        spots_manager.update_param('alpha', new_maxv=new)

spots_alpha_slider.on_change('value', update_spots_alpha)

## Color picker
spots_color_picker = ColorPicker(title="Select Spots Color", color=INITIAL_COLOR)
def update_spots_color(attr, old, new):
    print("Changing color")
    def update_color():
        spots_manager.update_param('color', new_val=new)

spots_color_picker.on_change('color', update_spots_color)

## Visibility checkbox
checkbox = CheckboxGroup(labels=["Show spots"], active=[0])

def on_checkbox_change(attr, old, new):
    visible = 0 in new
    circle_glyph.visible = visible
    spots_alpha_slider.disabled = not visible
    spots_color_picker.disabled = not visible

checkbox.on_change('active', on_checkbox_change)


def on_input_change(attr, old, new):
    print("Changing gene")
    def update_gene():
        spots_manager.update_colname(old, new)
        spots_manager.fetch_data()

gene_input.on_change('value', on_input_change)

175561it [00:09, 17687.90it/s]
100%|██████████| 77/77 [00:00<00:00, 173.18it/s]
/home/lalonso/PhD/VisHD/.venv/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/lalonso/PhD/VisHD/bokeh_server/../src/sample.py:100: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.loc[:,colname] = self.get_value(cropped_adata, cropped_gdf, colname)
/home/lalonso/PhD/VisHD/bokeh_server/manager.py:79: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/s

In [26]:
layout = row(p, column(img_alpha_slider, checkbox, spots_alpha_slider, spots_color_picker, gene_input))
show(p)